# Raport: Lista 1 – Zadanie 1
**Autor:** Bartosz Kruszewski 
**Data:** 2026-10-09

In [ ]:
#| include: false

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

Zdefiniujmy funkcje pomocniczne do wyliczania wag oraz metryk.

In [ ]:
def compute_theta4_weights(n):
    p = (np.arange(1, n + 1) - 0.5) / n
    z = stats.norm.ppf(p)
    w = stats.norm.pdf(z)
    return w / np.sum(w)


def compute_triangular_weights(n):
    w = np.arange(1, n + 1, dtype=float)
    w = np.minimum(w, n + 1 - w)
    return w / np.sum(w)


def calculate_metrics(estimates, true_val):
    bias = np.mean(estimates) - true_val
    var = np.var(estimates, ddof=1)
    mse = np.mean((estimates - true_val)**2)
    return bias, var, mse

Zdefinujmy estymatory zgodnie z wytycznymi z treści zadania.

In [ ]:
def est_mean(X):
    """(i) Średnia arytmetyczna."""
    return np.mean(X, axis=1)


def est_median(X):
    """(ii) Mediana."""
    return np.median(X, axis=1)


def est_weighted_custom(X):
    """(iii) Średnia ważona z własnymi wagami trójkątnymi."""
    n = X.shape[1]
    weights = compute_triangular_weights(n)
    return np.dot(X, weights)


def est_order_stats(X):
    """(iv) Średnia ważona statystyk pozycyjnych oparta na N(0,1)."""
    n = X.shape[1]
    X_sorted = np.sort(X, axis=1)
    weights = compute_theta4_weights(n)
    return np.dot(X_sorted, weights)


def est_harmonic_mean(X):
    """(v) Średnia harmoniczna: n / sum(1/X_i)."""
    n = X.shape[1]
    with np.errstate(divide='ignore', invalid='ignore'):
        inv_sum = np.sum(1.0 / X, axis=1)
        res = n / inv_sum
    return res


def est_power_mean_3(X):
    """(vi) Średnia potęgowa rzędu 3."""
    mean_cube = np.mean(X**3, axis=1)
    return np.sign(mean_cube) * np.abs(mean_cube)**(1.0 / 3.0)


def est_trimmed_mean(X, trim_ratio=0.10):
    """(vii) Estymator własny: średnia ucinana."""
    n = X.shape[1]
    k = int(trim_ratio * n)
    if k == 0:
        return np.mean(X, axis=1)
    X_sorted = np.sort(X, axis=1)
    return np.mean(X_sorted[:, k:n-k], axis=1)


estimators = [
    est_mean,
    est_median,
    est_weighted_custom,
    est_order_stats,
    est_harmonic_mean,
    est_power_mean_3,
    est_trimmed_mean
]

In [ ]:
#| include: false

def run_simulation(n, theta, sigma, M, seed=42):
    np.random.seed(seed)
    X = np.random.normal(loc=theta, scale=sigma, size=(M, n))

    estimators_dict = {f"t{i+1}: {est.__name__}": est(X) for i, est in enumerate(estimators)}
    
    records = [(name, *calculate_metrics(vals, theta)) for name, vals in estimators_dict.items()]
    df_metrics = pd.DataFrame(records, columns=['Estymator', 'Obciążenie (Bias)', 'Wariancja', 'MSE'])

    print(f"Metrics (n={n}, theta={theta}, sigma={sigma}):")
    display(df_metrics.round(5))

    ax = sns.boxplot(data=estimators_dict, showmeans=True)
    ax.axhline(theta, color='red', ls='--', label=rf'Prawdziwe $\theta = {theta}$')
    ax.set(title=rf"$n={n}, \theta={theta}, \sigma={sigma}$", xlabel='Estymator', ylabel='Wartość estymatora')
    ax.tick_params(axis='x', rotation=20)
    ax.legend()
    plt.tight_layout()
    plt.show()

Wygenerowane wykresy pudełkowe dla konkretnych wartości n, $\theta$ i $\sigma$ 

In [ ]:
#| echo: false
#| output: true

for n in [20, 100]:
    for theta in [0, 4]:
        for sigma in [1, 2]:
            run_simulation(n=n, theta=theta, sigma=sigma, M=10000)

Z przeprowadzonego eksperymentu wynika, że większość estymatorów w miarę dobrze odzwierciedla prawdziwe dane, natomiast średnia harmoniczna nie sprawdza się jako estymator w tym przypadku.